In [30]:
import ollama

In [31]:
def generate_llm_financial_interpretation(summary):

    prompt = f"""
You are a senior equity research analyst.

Interpret the following Apple financial analysis.

def interpret_liabilities_ratio(ratio):

    if ratio < 50:
        return "The company maintains relatively low liabilities compared to assets."

    elif ratio < 90:
        return "The company carries substantial liabilities relative to assets, but liabilities remain below total assets."

    else:
        return "The company carries very high liabilities relative to assets."
        
Rules:
1. Do not make claims not directly supported by the metrics.
2. Do not compare with industry averages unless benchmark data is provided.
3. Do not say liabilities exceed assets unless ratio > 100%.
4. Use cautious analyst-style language.
5. Separate observations from conclusions.

Focus on:
- profitability
- growth
- efficiency
- financial health

Keep the answer concise but insightful.
Do not compare with industry average unless benchmark data is provided.

Financial summary:
{summary}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    return response["message"]["content"]

In [32]:
import json

In [33]:
metrics_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

In [34]:
def get_value(metric_key, year):
    return metrics[metric_key]["values"][str(year)]


def percentage_change(current, previous):
    return ((current - previous) / previous) * 100


def margin(part, total):
    return (part / total) * 100


def safe_round(value):
    return round(value, 2)


def format_money(value):
    return f"${value:,.0f} million"


def format_percent(value):
    return f"{value:.2f}%"

In [35]:
def generate_financial_analysis(year=2023):
    
    previous_year = year - 1

    # Revenue
    revenue = get_value("total_net_sales", year)
    prev_revenue = get_value("total_net_sales", previous_year)

    revenue_growth = percentage_change(revenue, prev_revenue)

    # Net income
    net_income = get_value("net_income", year)
    prev_net_income = get_value("net_income", previous_year)

    net_income_growth = percentage_change(
        net_income,
        prev_net_income
    )

    # Margins
    gross_margin_value = get_value("gross_margin", year)
    operating_income = get_value("operating_income", year)

    gross_margin_pct = margin(
        gross_margin_value,
        revenue
    )

    operating_margin_pct = margin(
        operating_income,
        revenue
    )

    net_profit_margin_pct = margin(
        net_income,
        revenue
    )

    # Expenses
    rd = get_value("research_and_development", year)

    sga = get_value(
        "selling_general_and_administrative",
        year
    )

    total_opex = get_value(
        "total_operating_expenses",
        year
    )

    rd_pct = margin(rd, revenue)

    sga_pct = margin(sga, revenue)

    opex_pct = margin(total_opex, revenue)

    # Balance sheet
    assets = get_value("total_assets", year)

    liabilities = get_value(
        "total_liabilities",
        year
    )

    cash = get_value(
        "cash_and_cash_equivalents",
        year
    )

    liabilities_to_assets = margin(
        liabilities,
        assets
    )

    cash_to_assets = margin(
        cash,
        assets
    )

    analysis = {
        "year": year,

        "revenue": {
            "value": revenue,
            "growth_percent": safe_round(revenue_growth)
        },

        "net_income": {
            "value": net_income,
            "growth_percent": safe_round(net_income_growth)
        },

        "profitability": {
            "gross_margin_percent": safe_round(gross_margin_pct),
            "operating_margin_percent": safe_round(operating_margin_pct),
            "net_profit_margin_percent": safe_round(net_profit_margin_pct)
        },

        "expense_efficiency": {
            "rd_as_percent_of_sales": safe_round(rd_pct),
            "sga_as_percent_of_sales": safe_round(sga_pct),
            "operating_expenses_as_percent_of_sales": safe_round(opex_pct)
        },

        "balance_sheet": {
            "liabilities_to_assets_percent": safe_round(liabilities_to_assets),
            "cash_to_assets_percent": safe_round(cash_to_assets)
        }
    }

    return analysis

In [36]:
def detect_calculation_intent(question):
    q = question.lower()

    if "liabilities" in q and "assets" in q:
        return "liabilities_vs_assets"

    if "net profit margin" in q:
        return "net_profit_margin"

    if "gross margin" in q and "%" in q:
        return "gross_margin_percent"

    if "operating margin" in q:
        return "operating_margin"

    if "r&d" in q or "research and development" in q:
        if "%" in q or "percentage" in q:
            return "rd_as_percent_of_sales"
        return "research_and_development"

    if "revenue growth" in q or "sales growth" in q:
        return "revenue_growth"

    if "compare" in q and "net sales" in q:
        return "net_sales_comparison"

    if "liabilities to assets" in q:
        return "liabilities_to_assets"

    if "cash to assets" in q:
        return "cash_to_assets"

    return "rag_only"

In [37]:
def calculate_answer(question):
    intent = detect_calculation_intent(question)

    if intent == "liabilities_vs_assets":

        ratio = margin(
            get_value("total_liabilities", 2023),
            get_value("total_assets", 2023)
        )

        if ratio > 100:
            interpretation = (
            "Liabilities exceed total assets."
            )
        else:
            interpretation = (
            "Liabilities do not exceed total assets."
        )

        return {
            "intent": intent,
            "answer": interpretation,
            "ratio": ratio
        }

    if intent == "net_profit_margin":
        revenue = get_value("total_net_sales", 2023)
        net_income = get_value("net_income", 2023)
        result = margin(net_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's net profit margin in 2023 was {format_percent(result)}.",
            "calculation": f"Net profit margin = Net income / Total net sales × 100 = {format_money(net_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "net_income_2023": net_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "gross_margin_percent":
        revenue = get_value("total_net_sales", 2023)
        gross_margin = get_value("gross_margin", 2023)
        result = margin(gross_margin, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's gross margin percentage in 2023 was {format_percent(result)}.",
            "calculation": f"Gross margin % = Gross margin / Total net sales × 100 = {format_money(gross_margin)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "gross_margin_2023": gross_margin,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "operating_margin":
        revenue = get_value("total_net_sales", 2023)
        operating_income = get_value("operating_income", 2023)
        result = margin(operating_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's operating margin in 2023 was {format_percent(result)}.",
            "calculation": f"Operating margin = Operating income / Total net sales × 100 = {format_money(operating_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "operating_income_2023": operating_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "rd_as_percent_of_sales":
        revenue = get_value("total_net_sales", 2023)
        rd = get_value("research_and_development", 2023)
        result = margin(rd, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's R&D expense as a percentage of sales in 2023 was {format_percent(result)}.",
            "calculation": f"R&D as % of sales = Research and development / Total net sales × 100 = {format_money(rd)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "research_and_development_2023": rd,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "revenue_growth":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased by {format_percent(abs(pct_change))} in 2023 compared with 2022.",
            "calculation": f"Revenue growth = (2023 net sales - 2022 net sales) / 2022 net sales × 100 = ({format_money(revenue_2023)} - {format_money(revenue_2022)}) / {format_money(revenue_2022)} × 100 = {format_percent(pct_change)}",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "net_sales_comparison":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased from {format_money(revenue_2022)} in 2022 to {format_money(revenue_2023)} in 2023.",
            "calculation": f"Difference = {format_money(revenue_2023)} - {format_money(revenue_2022)} = {format_money(change)}. Percentage change = {format_percent(pct_change)}.",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "liabilities_to_assets":
        liabilities = get_value("total_liabilities", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(liabilities, assets)

        return {
            "intent": intent,
            "answer": f"Apple's liabilities-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Liabilities to assets = Total liabilities / Total assets × 100 = {format_money(liabilities)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "total_liabilities_2023": liabilities,
                "total_assets_2023": assets
            }
        }

    if intent == "cash_to_assets":
        cash = get_value("cash_and_cash_equivalents", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(cash, assets)

        return {
            "intent": intent,
            "answer": f"Apple's cash-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Cash to assets = Cash and cash equivalents / Total assets × 100 = {format_money(cash)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "cash_and_cash_equivalents_2023": cash,
                "total_assets_2023": assets
            }
        }

    return None

In [38]:
def answer_financial_question(question):

    # Step 1 — Try deterministic calculator
    calculation_result = calculate_answer(question)

    if calculation_result is not None:

        prompt = f"""
You are a financial analyst.

Use ONLY the provided calculation result.

Question:
{question}

Calculation Result:
{json.dumps(calculation_result, indent=4)}

Write a concise financial answer.

Rules:
1. Do not invent numbers.
2. Do not add unsupported claims.
3. Keep answer factual and concise.
"""

        response = ollama.chat(
            model="mistral",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": 0.1
            }
        )

        return response["message"]["content"]

    # Step 2 — fallback for unsupported questions
    return "Question type not supported yet."

In [39]:
print(
    answer_financial_question(
        "What was Apple's net profit margin in 2023?"
    )
)

 Apple's net profit margin in 2023 was 25.31%. This calculation is based on the net income of $96,995 million and total net sales of $383,285 million for that year.


In [42]:
test_questions = [
    {
        "question": "What was Apple's net profit margin in 2023?",
        "expected_keywords": ["25.31", "net income", "total net sales"]
    },
    {
        "question": "What was Apple's operating margin in 2023?",
        "expected_keywords": ["29.82", "operating income"]
    },
    {
        "question": "Did Apple's liabilities exceed assets?",
        "expected_keywords": ["do not exceed", "below 100"]
    },
    {
        "question": "Compare Apple's 2023 and 2022 net sales.",
        "expected_keywords": ["383,285", "394,328", "decreased", "2.8"]
    },
    {
        "question": "Did Apple's liabilities exceed assets?",
        "expected_keywords": ["do not exceed", "82.37"]
    }
]

In [43]:
for test in test_questions:
    response = answer_financial_question(test["question"])

    print("QUESTION:", test["question"])
    print("RESPONSE:", response)

    for keyword in test["expected_keywords"]:
        print(keyword, "=>", keyword.lower() in response.lower())

    print("-" * 100)

QUESTION: What was Apple's net profit margin in 2023?
RESPONSE:  Apple's net profit margin in 2023 was 25.31%. This calculation is based on the net income of $96,995 million divided by total net sales of $383,285 million for that year.
25.31 => True
net income => True
total net sales => True
----------------------------------------------------------------------------------------------------
QUESTION: What was Apple's operating margin in 2023?
RESPONSE:  Apple's operating margin in 2023 was 29.82%. This figure is calculated as the operating income ($114,301 million) divided by total net sales ($383,285 million), then multiplied by 100.
29.82 => True
operating income => True
----------------------------------------------------------------------------------------------------
QUESTION: Did Apple's liabilities exceed assets?
RESPONSE:  Based on the provided calculation result, Apple's liabilities do not exceed total assets. The liabilities-to-assets ratio is 82.37%.
do not exceed => True
be